# Movie Recommendation System Notebook

This is an exploratory / learning notebook for building a simple movie recommendation system.

- Dataset: MovieLens
- Approach: KNN with cosine similarity
- Filtering: top **2500 most-rated movies** to keep the matrix manageable

Note: the full original dataset is not included in the repository because it is large.

In [1]:
# Import the libraries needed for data handling, sparse matrices, and KNN.
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

from scipy.sparse import csr_matrix

from sklearn.neighbors import NearestNeighbors

## 1. Load the datasets

First, load the movies and ratings files and do a few quick checks such as `head()` and `info()`.

In [14]:
# Load the movie titles/metadata file.
mv_names_df=pd.read_csv('movies.csv')

In [15]:
mv_names_df.head(5)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [16]:
mv_names_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87585 entries, 0 to 87584
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  87585 non-null  int64 
 1   title    87585 non-null  object
 2   genres   87585 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.0+ MB


In [17]:
mv_names_df.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [18]:
# Load the user ratings file.
mv_rating_df=pd.read_csv('ratings.csv')

In [19]:
mv_rating_df.head(5)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [21]:
mv_rating_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [22]:
mv_rating_df.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

## 2. Select the most-rated movies

To reduce memory usage, the notebook keeps only the most frequently rated movies.

In [28]:
mv_rating_df['movieId'].value_counts()

318       102929
356       100296
296        98409
2571       93808
593        90330
           ...  
288825         1
288467         1
287221         1
284087         1
274343         1
Name: movieId, Length: 84432, dtype: int64

In [23]:
# Keep only the most frequently rated movies to reduce the matrix size.
top_ids = mv_rating_df['movieId'].value_counts().head(2500).index.tolist()

In [24]:
top_ids[:5]

[318, 356, 296, 2571, 593]

In [26]:
mv_names_df[mv_names_df['movieId'].isin(top_ids[:5])]

,movieId,title,genres
292,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
314,318,"Shawshank Redemption, The (1994)",Crime|Drama
351,356,Forrest Gump (1994),Comedy|Drama|Romance|War
585,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller


In [32]:
# Filter the movies table so it only contains the selected top movies.
top_movies=mv_names_df[mv_names_df['movieId'].isin(top_ids)].copy()

In [33]:
top_movies.tail(5)

,movieId,title,genres
76867,254726,Dune (2021),Action|Adventure|Drama|Sci-Fi
78698,263007,Spider-Man: No Way Home (2021),Action|Adventure|Fantasy|Sci-Fi
80120,268642,The Batman (2022),Action|Crime|Drama
80669,270698,Everything Everywhere All at Once (2022),Action|Comedy|Sci-Fi
81572,274053,Top Gun: Maverick (2022),Action|Drama


## 3. Merge data and build the movie-user matrix

Here, ratings are merged with movie titles and reshaped into a matrix.

In [48]:
# Merge ratings with movie titles so each rating has its movie name.
top_movies_ratings = pd.merge(mv_rating_df, top_movies, on='movieId',how='inner')

In [49]:
top_movies_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,17,4.0,944249077,Sense and Sensibility (1995),Drama|Romance
1,3,17,5.0,1084485217,Sense and Sensibility (1995),Drama|Romance
2,15,17,4.5,1289858271,Sense and Sensibility (1995),Drama|Romance
3,28,17,4.0,961513829,Sense and Sensibility (1995),Drama|Romance
4,29,17,4.0,845056111,Sense and Sensibility (1995),Drama|Romance


In [50]:
top_movies_ratings.shape

(25506873, 6)

In [52]:
top_movies_ratings[top_movies_ratings['userId']==1].head(10)

,userId,movieId,rating,timestamp,title,genres
0,1,17,4.0,944249077,Sense and Sensibility (1995),Drama|Romance
22251,1,25,1.0,944250228,Leaving Las Vegas (1995),Drama|Romance
44776,1,29,2.0,943230976,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
54189,1,32,5.0,943228858,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
109464,1,34,2.0,943228491,Babe (1995),Children|Drama
145433,1,36,1.0,944249008,Dead Man Walking (1995),Crime|Drama
167192,1,110,3.0,943231119,Braveheart (1995),Action|Drama|War
236674,1,111,5.0,944249008,Taxi Driver (1976),Crime|Drama|Thriller
273321,1,161,1.0,943231162,Crimson Tide (1995),Drama|Thriller|War
298591,1,176,4.0,944079496,Living in Oblivion (1995),Comedy


In [54]:
# Build a movie-user matrix where rows are movies and columns are users.
movie_user_matrix = top_movies_ratings.pivot_table(
    index='title',
    columns='userId',
    values='rating'
    ).fillna(0)

In [55]:
movie_user_matrix.shape

(2500, 200937)

In [60]:
movie_user_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,200939,200940,200941,200942,200943,200944,200945,200946,200947,200948
title,,,,,,,,,,,,,,,,,,,,,
"'burbs, The (1989)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
(500) Days of Summer (2009),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
10 Cloverfield Lane (2016),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
10 Things I Hate About You (1999),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
"10,000 BC (2008)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0


In [61]:
# Store movie titles in row order so we can map matrix rows back to names.
movie_names = movie_user_matrix.index.tolist()

In [63]:
len(movie_names)

2500

In [64]:
# Create a fast lookup from movie title to row index.
movie_to_row = {movie: i for i, movie in enumerate(movie_names)}

## 4. Convert to sparse format and train the model

The matrix is converted to CSR format and then used to train a KNN model.

In [62]:
# Convert the dense matrix into CSR sparse format for memory efficiency.
movie_user_csr = csr_matrix(movie_user_matrix.values)

In [57]:
# Train a KNN model using cosine distance on the sparse movie-user matrix.
model_knn = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=10)
model_knn.fit(movie_user_csr)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [65]:
# Return the closest movies to a given title.
def recommend_movies(movie_name, csr_data, movie_names_data, movie_to_row, model, n_recommendations=5):
    if movie_name not in movie_to_row:
        raise ValueError("Movie not found in the filtered dataset.")

    movie_idx = movie_to_row[movie_name]
    movie_vector = csr_data[movie_idx]

    distances, indices = model.kneighbors(movie_vector, n_neighbors=n_recommendations + 1)

    recommendations = []

    for rec_idx, rec_distance in zip(indices.flatten()[1:], distances.flatten()[1:]):
        recommendations.append((movie_names_data[rec_idx], rec_distance))

    return recommendations

## 5. Test the recommender

Run a couple of example queries to check whether the model returns reasonable results.

In [66]:
recommend_movies("Toy Story (1995)",movie_user_csr,movie_names,movie_to_row,model_knn,5)

[('Toy Story 2 (1999)', 0.42518556082057335),
 ('Star Wars: Episode IV - A New Hope (1977)', 0.43828935661092683),
 ('Back to the Future (1985)', 0.45447070280177737),
 ('Forrest Gump (1994)', 0.45841994638468053),
 ('Jurassic Park (1993)', 0.460095613442843)]

In [67]:
recommend_movies("Shutter Island (2010)",movie_user_csr,movie_names,movie_to_row,model_knn,5)

[('Inception (2010)', 0.3825581031116244),
 ('Django Unchained (2012)', 0.4354113482132532),
 ('Inglourious Basterds (2009)', 0.44007889476376394),
 ('Interstellar (2014)', 0.4474342010121479),
 ('Prestige, The (2006)', 0.447987098056814)]

## 6. Save the trained artifacts

Export the sparse matrix, lookup structures, and trained model into a pickle file for the web app.

In [ ]:
# Save the trained objects so they can be loaded by the FastAPI app later.
import pickle

with open("recommender_data.pkl", "wb") as f:
    pickle.dump({
        "csr_data": movie_user_csr,
        "movie_names": movie_names,
        "movie_to_row": movie_to_row,
        "model": model_knn
    }, f)